# Time Series Forecasting
## Predicting Future Values from Historical Data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

## 1. Generate Time Series Data

In [ ]:
# Create synthetic time series
np.random.seed(42)
t = np.arange(0, 100, 0.1)
values = 50 + 20*np.sin(t/5) + np.random.normal(0, 2, len(t))

df = pd.DataFrame({'time': t, 'value': values})

plt.figure(figsize=(12, 5))
plt.plot(df['time'], df['value'])
plt.title('Time Series Data')
plt.xlabel('Time')
plt.ylabel('Value')
plt.show()

## 2. Moving Average

In [ ]:
df['MA_5'] = df['value'].rolling(window=5).mean()
df['MA_20'] = df['value'].rolling(window=20).mean()

plt.figure(figsize=(12, 5))
plt.plot(df['time'], df['value'], label='Original', alpha=0.7)
plt.plot(df['time'], df['MA_5'], label='MA(5)', linewidth=2)
plt.plot(df['time'], df['MA_20'], label='MA(20)', linewidth=2)
plt.legend()
plt.title('Moving Averages')
plt.show()

## 3. Train-Test Split for Time Series

In [ ]:
# Create features and targets
lookback = 10
X_ts = []
y_ts = []

for i in range(len(df) - lookback):
    X_ts.append(df['value'].values[i:i+lookback])
    y_ts.append(df['value'].values[i+lookback])

X_ts = np.array(X_ts)
y_ts = np.array(y_ts)

# Split into train and test
split_idx = int(len(X_ts) * 0.8)
X_train, X_test = X_ts[:split_idx], X_ts[split_idx:]
y_train, y_test = y_ts[:split_idx], y_ts[split_idx:]

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

## 4. Simple Forecasting Models

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Prepare data for sklearn (flatten)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# Train models
lr = LinearRegression()
lr.fit(X_train_flat, y_train)
y_pred_lr = lr.predict(X_test_flat)

rf_ts = RandomForestRegressor(n_estimators=100, random_state=42)
rf_ts.fit(X_train_flat, y_train)
y_pred_rf = rf_ts.predict(X_test_flat)

# Evaluate
print('Linear Regression MSE:', mean_squared_error(y_test, y_pred_lr))
print('Random Forest MSE:', mean_squared_error(y_test, y_pred_rf))

## 5. Visualization

In [ ]:
plt.figure(figsize=(14, 6))

# Get actual test indices
test_indices = np.arange(split_idx + lookback, len(df))

plt.plot(test_indices, y_test, label='Actual', linewidth=2, marker='o')
plt.plot(test_indices, y_pred_lr, label='Linear Regression Forecast', linewidth=2, marker='^')
plt.plot(test_indices, y_pred_rf, label='Random Forest Forecast', linewidth=2, marker='s')

plt.legend()
plt.title('Time Series Forecasting')
plt.xlabel('Time Index')
plt.ylabel('Value')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()